# NeuroPlast — final sweep (one click: Runtime → Run all)

Runs the same experiment protocol as the dev sessions with **extra seeds**, so the
RQ answers have real statistical power. Use a **CPU runtime** (High-RAM / more
cores if you have Colab Pro). GPU/TPU don't help: the networks are tiny and time
goes into environment stepping; throughput = number of CPU cores.

Everything writes into `runs/` on your Google Drive, so if the runtime
disconnects, just **Run all again**: finished jobs are skipped, half-finished
ones restart. When it finishes, download `neuroplast_results.zip` (or leave it
on Drive) and share it / commit `runs/` + `results/`.

In [ ]:
# 1. Settings
BRANCH = "main"
SEEDS = "4-10"         # dev session used seeds 1-3; these add 7 more
JOB_FILES = ["rq1_v2", "cl_budget", "cl_snn", "cl_cnn"]   # priority order: RQ1 v2 on 6x6 is bimodal
# per seed (solve / never), so it needs the most seeds; cl_snn is the slowest
USE_DRIVE = True

In [ ]:
# 2. Code + dependencies (if the repo is private: add a GitHub token as Colab secret GH_TOKEN)
import os, subprocess
try:
    from google.colab import userdata
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
url = f"https://{tok}@github.com/VT69/NeuroPlast.git" if tok else "https://github.com/VT69/NeuroPlast.git"
if not os.path.exists("/content/NeuroPlast"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/NeuroPlast"], check=True)
%cd /content/NeuroPlast
!pip install -q minigrid gymnasium snntorch pandas pyyaml tabulate
!python -c "import torch, minigrid; print(torch.__version__, minigrid.__version__)"
!nproc

In [ ]:
# 3. Persist runs/ on Drive (resumable across disconnects)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/neuroplast_runs"
    os.makedirs(target, exist_ok=True)
    # copy the dev-session runs once so analysis sees all seeds together
    !rsync -a --ignore-existing runs/ {target}/
    !rm -rf runs && ln -s {target} runs
!ls runs | head

In [ ]:
# 4. Generate extra-seed job files and run them on all cores
!python scripts/make_jobs.py --seeds {SEEDS} --suffix _more
files = " ".join(f"jobs/{f}_more.txt" for f in JOB_FILES)
!OMP_NUM_THREADS=1 bash scripts/worker.sh $(nproc) {files} 2>&1 | grep -E "START|END|pending"

In [ ]:
# 5. Aggregate (tables in results/*.md, figures in results/figs) and package
!python scripts/analyze.py all > results/analyze_stdout.txt; tail -40 results/analyze_stdout.txt
!zip -qr neuroplast_results.zip results runs -x "runs/**/data.pt" "runs/**/*.lock"
from google.colab import files as _f
_f.download("neuroplast_results.zip")